<a href="https://colab.research.google.com/github/akifnu/DSprojects/blob/main/perfumerecom/Perfume_Recommender_Decision1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🟢 Perfume Recommender v2: Microsoft Decision-1 Edition

This is the second version of the Perfume Recommender. The idea is the same: describe a memory, a mood or a moment, and get five perfumes that fit it. The ranking step is new.

**What changed from v1**

| | v1 | v2 (this notebook) |
|---|---|---|
| Embedding model | `intfloat/multilingual-e5-large-instruct` (560M) | `microsoft/harrier-oss-v1-270m` (268M): about half the size, and higher on MTEB multilingual (66.6 vs 63.2) and retrieval (66.4 vs 57.1) |
| Search | FAISS | Plain numpy: one matrix multiply over a few thousand perfumes |
| Shortlist | 25 candidates | 15 candidates |
| Ranking model | Llama 4 Maverick on Groq, via `aisuite` | `microsoft/microsoft-decision-1` on **OpenRouter** |
| How the panel answers | The LLM is asked to "privately" run 5 experts and print 5 numbers, which are then parsed with a regex | Each of the 5 experts is a separate **scored question**, and the model returns a calibrated probability for every score level |
| Output | 5 candidate numbers | A 0–4 score from every expert for every candidate, plus the consensus score |

Microsoft-Decision-1 does not write text. It reads the content you give it and returns a probability for each answer option you define. That suits this task: it judges how well each of the 15 shortlisted perfumes fits your request, from each expert's point of view, and the scores are numbers you can read rather than text you have to parse.

To try it, clone the repo:

```bash
git clone https://github.com/akifnu/DSprojects.git
cd DSprojects/perfumerecom
```

## ⚙️ Environment Setup

Only the packages this notebook uses are installed. `groq`, `aisuite` and `langchain` are no longer needed, because the OpenRouter Decisions API is a plain HTTPS call made with `requests`.

FAISS is no longer needed either. It's built for searching millions of items, and with a few thousand perfumes a single numpy matrix multiply is just as fast. `-U` makes sure `transformers` is recent enough for Harrier, which is built on Gemma 3.

In [ ]:
%pip install -q -U sentence-transformers transformers openpyxl requests pandas

## 🔑 OpenRouter API Key

Create a key at [openrouter.ai/keys](https://openrouter.ai/keys). The notebook looks for it in this order:

1. A Colab secret named `Decision_api` (🔑 icon in the Colab sidebar, with "Notebook access" turned on). This is the recommended option.
2. A `Decision_api` environment variable.
3. A hidden prompt.

The older name `OPENROUTER_API_KEY` also works, as a secret or an environment variable.

**Never paste the key into a code cell.** Notebooks are saved with their cell contents, so a pasted key gets committed to GitHub along with the code.

In [ ]:
import os
import getpass

KEY_NAMES = ["Decision_api", "OPENROUTER_API_KEY"]  # first one found wins

def get_openrouter_key():
    for name in KEY_NAMES:
        try:
            from google.colab import userdata  # only exists on Colab
            key = userdata.get(name)
            if key:
                return key
        except Exception:
            pass
        if os.environ.get(name):
            return os.environ[name]
    return getpass.getpass("Enter your OpenRouter API key: ")

OPENROUTER_API_KEY = get_openrouter_key()

## 🗂️ Load the Perfume Data

The filtering is the same as in v1 (brands with at least 60 perfumes, released in 2000 or later, rated 4.0 or higher). The cached embeddings file has one row per perfume in this filtered table, so changing the filters requires rebuilding it. The notebook rebuilds it automatically when the row count no longer matches.

The data file can be `fragrantica.xlsx` next to this notebook, or `/content/Fragrantica.xlsx` on Colab (upload it from the Files panel).

In [ ]:
import re
import numpy as np
import pandas as pd

DATA_CANDIDATES = ["fragrantica.xlsx", "Fragrantica.xlsx", "/content/Fragrantica.xlsx", "/content/fragrantica.xlsx"]
data_path = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError(
        "Couldn't find the Fragrantica Excel file. Upload it next to this notebook "
        "(or to /content on Colab). Source: https://www.kaggle.com/datasets/olgagmiufana1/fragrantica-com-fragrance-dataset"
    )

df = pd.read_excel(data_path)
df.columns = df.columns.str.strip()

brand_counts = df["Brand"].value_counts()
brands_to_keep = brand_counts[brand_counts >= 60].index
df_filtered = df[df["Brand"].isin(brands_to_keep)].copy()

df_filtered = df_filtered.dropna(subset=["Year"])
df_filtered["Year"] = df_filtered["Year"].astype(int)
df_filtered = df_filtered[df_filtered["Year"] >= 2000]

df_filtered["Rating Value"] = (
    df_filtered["Rating Value"].astype(str).str.replace(",", ".", regex=False).astype(float)
)
df_filtered = df_filtered[df_filtered["Rating Value"] >= 4]
df_filtered = df_filtered.reset_index(drop=True)

def combine_notes_accord(row):
    notes = row.get("Notes", "N/A")
    accord = row.get("Accord", "N/A")
    brand = row.get("Brand", "N/A")
    gender = row.get("Gender", "N/A")
    return f"Notes: {notes}; Accord: {accord}; Brand: {brand}; Perfume's Gender (unisex should be both men and women): {gender}"

df_filtered["Description"] = df_filtered.apply(combine_notes_accord, axis=1)
print(f"{len(df_filtered):,} perfumes after filtering (from {data_path})")

## 🔎 Embeddings and Search

[Harrier OSS v1 270M](https://huggingface.co/microsoft/harrier-oss-v1-270m) turns each perfume's notes and accords, and your request, into 640 numbers that capture meaning. Perfumes and requests that mean similar things end up close together, even when they share no words.

Harrier is asymmetric: your request gets a short instruction in front of it (`Instruct: …
Query: …`), and the perfume descriptions are embedded as they are. The first run embeds the whole catalog once, which takes a minute or two on a GPU, and saves the result to `perfume_embeddings_harrier270m.npy` so later runs load it instantly. The v1 files (`perfume_embeddings.npy`, `faiss_index.bin`) were made with e5 and aren't used here.

The search is one matrix multiply: the vectors are normalised, so the dot product is the cosine similarity. It returns the 15 perfumes closest to your request, and those become the candidates the expert panel scores.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

EMBED_MODEL = "microsoft/harrier-oss-v1-270m"
embedder = SentenceTransformer(EMBED_MODEL, device=device)

# Harrier embeds queries with a task instruction; documents get none.
QUERY_PROMPT = (
    "Instruct: Given a description of a desired perfume (a mood, memory, occasion, season or notes), "
    "retrieve perfumes whose notes and accords match it\nQuery: "
)

EMBEDDINGS_PATH = "perfume_embeddings_harrier270m.npy"

doc_embeddings = None
if os.path.exists(EMBEDDINGS_PATH):
    doc_embeddings = np.load(EMBEDDINGS_PATH)
    if doc_embeddings.shape[0] != len(df_filtered):
        print(f"Cached embeddings have {doc_embeddings.shape[0]} rows but the data has {len(df_filtered)}; rebuilding.")
        doc_embeddings = None
    else:
        print("Loaded cached embeddings.")

if doc_embeddings is None:
    doc_embeddings = embedder.encode(
        df_filtered["Description"].tolist(), batch_size=32, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True,
    ).astype("float32")
    np.save(EMBEDDINGS_PATH, doc_embeddings)
    print(f"Embedded {len(doc_embeddings):,} perfumes and cached them to {EMBEDDINGS_PATH}.")

def retrieve_candidates(user_query, top_n=15):
    query_embedding = embedder.encode(
        [user_query], prompt=QUERY_PROMPT, convert_to_numpy=True, normalize_embeddings=True
    )[0].astype("float32")
    sims = doc_embeddings @ query_embedding               # cosine similarity to every perfume
    top_n = min(top_n, len(sims))
    top = np.argpartition(-sims, top_n - 1)[:top_n]       # the top_n best, unordered
    top = top[np.argsort(-sims[top])]                     # sorted best first
    candidates = df_filtered.iloc[top].copy().reset_index(drop=True)
    candidates["Similarity"] = sims[top]
    return candidates

## 🧑‍⚖️ The Expert Panel on Microsoft-Decision-1

v1 asked one LLM to role-play five experts in its head. Here each expert is a separate **score question** sent to Microsoft-Decision-1 through OpenRouter's Decisions API (`POST https://openrouter.ai/api/alpha/decisions`).

For every candidate perfume the request contains:

- `state`: your request plus everything the dataset knows about that perfume.
- `questions`: the five experts. Each scores the perfume on the same five-level rubric (0 = clear mismatch … 4 = excellent fit), judged from its own perspective.

For every expert the model returns a probability for each level. The notebook reduces that to an **expected score**, Σ level × probability, so a perfume the expert is unsure about lands between levels instead of being forced into one. The **consensus** is the average of the five experts. The Synthesizer, whose job is the overall fit, breaks ties, and embedding similarity breaks any that remain.

The 15 candidates are scored in parallel. Microsoft-Decision-1 is listed at about $0.042 per million input tokens, with output free, so one search costs a small fraction of a cent.

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor

DECISIONS_URL = "https://openrouter.ai/api/alpha/decisions"
DECISION_MODEL = "microsoft/microsoft-decision-1"

# Lowest level first; each level is self-contained, as the Decisions API expects.
FIT_LEVELS = [
    "Clear mismatch: this perfume goes against what the request asks for from this expert's perspective.",
    "Weak fit: only a small part matches; the most important aspects for this expert are missing.",
    "Partial fit: some important aspects match, but others are missing or uncertain.",
    "Good fit: most aspects this expert cares about match, with only minor gaps.",
    "Excellent fit: matches everything this expert cares about in the request.",
]

EXPERTS = {
    "literal": "You are the Literal expert. Judge only the explicit requirements in the user's request: named notes, "
               "accords, gender, brand, price or other stated constraints. Ignore anything the user did not say outright.",
    "emotional": "You are the Emotional expert. Judge whether this perfume's notes and accords would create the mood, "
                 "feeling, memory or vibe the user is describing.",
    "contextual": "You are the Contextual expert. Judge whether this perfume suits the season, occasion, setting or "
                  "lifestyle the request states or implies (e.g. summer beach day, office, evening party).",
    "resolver": "You are the Resolver expert. When the request contains conflicting or ambiguous wishes, judge how well "
                "this perfume balances them in the most sensible reading of what the user wants. If there is no "
                "conflict, judge the overall plausibility of the match.",
    "synthesizer": "You are the Synthesizer expert. Weigh the literal requirements, the emotion, the context and any "
                   "trade-offs together and judge how well this perfume fits the user's request overall.",
}

PANEL_QUESTIONS = {
    name: {"type": "score", "instructions": instructions, "criteria": FIT_LEVELS}
    for name, instructions in EXPERTS.items()
}

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
    # Optional: identifies the app on openrouter.ai
    "X-Title": "Perfume Recommender",
})

def perfume_record(row):
    """Everything the dataset knows about one perfume, as JSON-safe strings."""
    return {
        str(col): str(val) for col, val in row.items()
        if col not in ("Similarity", "Description") and pd.notna(val) and str(val).strip()
    }

def expected_score(answer):
    """Probability-weighted score on the 0-4 scale (falls back to the reported score)."""
    probs = answer.get("probabilities") or {}
    if probs:
        total = sum(probs.values()) or 1.0
        return sum(int(level) * p for level, p in probs.items()) / total
    return float(answer["score"])

def score_candidate(user_query, row, retries=1, timeout=30):
    body = {
        "model": DECISION_MODEL,
        "state": {"user_request": user_query, "perfume": perfume_record(row)},
        "questions": PANEL_QUESTIONS,
    }
    for attempt in range(retries + 1):
        try:
            resp = session.post(DECISIONS_URL, json=body, timeout=timeout)
            if resp.status_code in (429, 500, 502, 503, 504) and attempt < retries:
                continue
            resp.raise_for_status()
            answers = resp.json()["answers"]
            return {name: expected_score(answers[name]) for name in EXPERTS}
        except Exception as err:
            if attempt == retries:
                print(f"⚠️ Could not score {row.get('Perfume', 'candidate')}: {err}")
    return {name: np.nan for name in EXPERTS}

def rerank_with_decision1(user_query, candidates_df, final_n=5, max_workers=8):
    rows = [row for _, row in candidates_df.iterrows()]
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        scores = list(pool.map(lambda r: score_candidate(user_query, r), rows))

    scored = candidates_df.copy()
    for name in EXPERTS:
        scored[name.capitalize()] = [s[name] for s in scores]
    expert_cols = [n.capitalize() for n in EXPERTS]
    scored["Consensus"] = scored[expert_cols].mean(axis=1)

    if scored["Consensus"].isna().all():
        print("⚠️ The panel could not score any candidate; falling back to embedding similarity order.")
        return scored.head(final_n)

    scored = scored.sort_values(
        ["Consensus", "Synthesizer", "Similarity"], ascending=False, na_position="last"
    )
    return scored.head(final_n).reset_index(drop=True)

## 🟢 Try It

Describe the scent you want in your own words, for example *"the first warm evening of summer on a terrace by the sea"* or *"cozy, sweet and a bit smoky for winter nights, unisex"*.

The table shows the five recommendations with each expert's score (0–4) and the consensus.

In [ ]:
SHOW_COLS = ["Perfume", "Brand", "Gender", "Year", "Rating Value", "Notes", "Accord",
             "Literal", "Emotional", "Contextual", "Resolver", "Synthesizer", "Consensus"]

user_query = input("Describe your ideal perfume: ").strip()
candidates = retrieve_candidates(user_query, top_n=15)

if candidates.empty:
    print("No matching perfumes found.")
else:
    top5 = rerank_with_decision1(user_query, candidates, final_n=5)
    cols = [c for c in SHOW_COLS if c in top5.columns]
    print("\nTop 5 recommendations:\n")
    display(top5[cols].round(2))